## Chargement des Données via SQLAlchemy

Configurer la connexion SQLAlchemy

In [63]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

if not all([user, password, db]):
    raise ValueError("One or more DB environment variables are missing! Check your .env file.")

# connection string
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{db}")

# test connection
try:
    with engine.connect() as conn:
        print('Connection successful!')
except Exception as e:
    print(f'connection failed: {e}')

Connection successful!


Infrastructure: Automated Database Setup

In [64]:

def run_sql_file(filename, connection):
    from sqlalchemy import text
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            sql_command = f.read()
        with connection.begin() as conn:
            conn.execute(text(sql_command))
        print(f"✅ {filename} executed successfully!")
    except FileNotFoundError:
        print(f"❌ Error: {filename} not found. Check your file path!")

# This builds your entire database structure in order
scripts = [
    "../sql/drop_tables.sql",
    "../sql/create_tables.sql", 
    "../sql/add_constraints.sql", 
    "../sql/performance.sql"
]
for script in scripts:
    run_sql_file(script, engine)

✅ ../sql/drop_tables.sql executed successfully!
✅ ../sql/create_tables.sql executed successfully!
✅ ../sql/add_constraints.sql executed successfully!
✅ ../sql/performance.sql executed successfully!


Séparer financecore_clean.csv selon les tables normalisées avant insertion


In [65]:
import pandas as pd
df = pd.read_csv("../data/financecore_clean.csv")

print(df.head())
print(df.columns)

  transaction_id client_id     date_transaction  annee  mois  trimestre  \
0      TXN000559   CLI0060  2022-04-19 02:31:00   2022     4          2   
1      TXN001154   CLI0057  2024-06-20 20:51:00   2024     6          2   
2      TXN000764   CLI0015  2024-08-28 05:03:00   2024     8          3   
3      TXN001598   CLI0045  2024-01-07 08:16:00   2024     1          1   
4      TXN001873   CLI0034  2024-08-11 19:52:00   2024     8          3   

  jour_semaine      categorie              produit type_operation  ...  \
0      Tuesday  Depot especes       Compte Epargne         Credit  ...   
1     Thursday    Retrait DAB  Credit Consommation          Debit  ...   
2    Wednesday    Prelevement                  PEA          Debit  ...   
3       Sunday    Paiement CB  Credit Consommation         Credit  ...   
4       Sunday       Interets    Credit Immobilier         Credit  ...   

   montant devise  taux_change_eur montant_eur  montant_eur_verifie  \
0  2050.42    EUR             1.0

In [66]:
df['date_transaction'] = pd.to_datetime(df['date_transaction'])

# 2. Dimensions
agencies_df = df[['agence']].drop_duplicates().reset_index(drop=True)
products_df = df[['produit']].drop_duplicates().reset_index(drop=True)
categories_df = df[['categorie']].drop_duplicates().reset_index(drop=True)
clients_df = df[['client_id', 'score_credit_client', 'categorie_risque', 'segment_client']].drop_duplicates(subset=['client_id'])

# 3. Time Dimension (Matching your 'dim_date' table)
dim_date = pd.DataFrame({'date_transaction': df['date_transaction'].unique()})
dim_date['annee'] = dim_date['date_transaction'].dt.year
dim_date['mois'] = dim_date['date_transaction'].dt.month
dim_date['trimestre'] = dim_date['date_transaction'].dt.quarter
dim_date['jour_semaine'] = dim_date['date_transaction'].dt.day_name()

print("Step 'Séparer financecore_clean.csv' is complete!")
print(f'- {len(agencies_df)} Unique agencies')
print(f'- {len(products_df)} Unique products')
print(f'- {len(categories_df)} Unique categories')
print(f'- {len(clients_df)} Unique clients')

Step 'Séparer financecore_clean.csv' is complete!
- 9 Unique agencies
- 8 Unique products
- 8 Unique categories
- 150 Unique clients


Insérer les données

In [67]:
# A. Safe insertion function
def safe_insert(dataframe, table_name, engine):
    try:
        dataframe.to_sql(table_name, engine, if_exists='append', index=False)
        print(f"✅ {table_name}: Inserted.")
    except Exception:
        print(f"ℹ️ {table_name}: Already contains data, skipping.")

# B. Insert Dimensions
safe_insert(agencies_df, 'agencies', engine)
safe_insert(products_df, 'products', engine)
safe_insert(categories_df, 'categories', engine)
safe_insert(clients_df, 'clients', engine)
safe_insert(dim_date, 'dim_date', engine)

# C. Map IDs for the Transactions Table
agencies_db = pd.read_sql("SELECT agence_id, agence FROM agencies", engine)
products_db = pd.read_sql("SELECT produit_id, produit FROM products", engine)
categories_db = pd.read_sql("SELECT categorie_id, categorie FROM categories", engine)
dates_db = pd.read_sql("SELECT date_id, date_transaction FROM dim_date", engine)

# D. Merge logic
df_mapped = df.merge(agencies_db, on='agence') \
              .merge(products_db, on='produit') \
              .merge(categories_db, on='categorie') \
              .merge(dates_db, on='date_transaction')

# E. Select exactly the 14 columns defined in your 'create_tables.sql'
transactions_df = df_mapped[[
    'transaction_id', 'client_id', 'agence_id', 'produit_id', 'categorie_id', 
    'date_id', 'montant', 'type_operation', 'devise', 'taux_change_eur', 
    'montant_eur_verifie', 'statut', 'is_anomaly', 'solde_avant'
]]

# F. Final Insertion
try:
    transactions_df.to_sql('transactions', engine, if_exists='append', index=False)
    print(f"🚀 SUCCESS: {len(transactions_df)} transactions inserted!")
except Exception as e:
    print(f"❌ Final insertion failed: {e}")

✅ agencies: Inserted.
✅ products: Inserted.
✅ categories: Inserted.
✅ clients: Inserted.
✅ dim_date: Inserted.
🚀 SUCCESS: 2000 transactions inserted!


Implémenter la gestion des conflits (ON CONFLICT DO NOTHING / DO UPDATE)

In [68]:
from sqlalchemy import text

# 1. DE-DUPLICATE in Python first (Crucial step to avoid CardinalityViolation)
# This keeps only the LATEST version of each transaction ID in your current batch
clean_batch_df = transactions_df.drop_duplicates(subset=['transaction_id'], keep='last')

# 2. Upload the clean batch to a temporary table
clean_batch_df.to_sql('temp_transactions', engine, if_exists='replace', index=False)

# 3. Execute the UPSERT
upsert_sql = text("""
INSERT INTO transactions (
    transaction_id, client_id, agence_id, produit_id, categorie_id, 
    date_id, montant, type_operation, devise, taux_change_eur, 
    montant_eur_verifie, statut, is_anomaly, solde_avant
)
SELECT * FROM temp_transactions
ON CONFLICT (transaction_id) 
DO UPDATE SET 
    statut = EXCLUDED.statut,
    is_anomaly = EXCLUDED.is_anomaly,
    solde_avant = EXCLUDED.solde_avant,
    montant = EXCLUDED.montant,
    montant_eur_verifie = EXCLUDED.montant_eur_verifie;

-- Clean up
DROP TABLE temp_transactions;
""")

with engine.begin() as conn:
    conn.execute(upsert_sql)
    print(f"✅ Successfully processed {len(clean_batch_df)} unique transactions.")

✅ Successfully processed 2000 unique transactions.


Vérifier l'intégrité référentielle après chargement (COUNT…..)

In [69]:
# 1. Ensure Python count only includes UNIQUE transaction IDs
# This removes the duplicates causing the '4000' count in your screenshot
final_unique_df = transactions_df.drop_duplicates(subset=['transaction_id'])

# 2. Re-run the Integrity Check
csv_count = len(final_unique_df)
db_count = pd.read_sql("SELECT COUNT(*) FROM transactions", engine).iloc[0,0]

print("--- Final Sync Report ---")
print(f"Unique Transactions (Python): {csv_count}")
print(f"Total Transactions (Database): {db_count}")

if db_count == csv_count:
    print("✅ Perfect Sync: Database matches unique CSV records.")
else:
    print(f"ℹ️ Total records in DB: {db_count}. Your current batch has {csv_count} unique rows.")

--- Final Sync Report ---
Unique Transactions (Python): 2000
Total Transactions (Database): 2000
✅ Perfect Sync: Database matches unique CSV records.


## Requêtes SQL Analytiques Avancées

Agrégations : total et moyenne des transactions par agence, produit et mois (GROUP BY, HAVING)

In [70]:
# Query the performance view created in performance.sql
query_performance = "SELECT * FROM vw_analytics_performances;"
df_performance = pd.read_sql(query_performance, engine)

# Display the top performers
df_performance.sort_values(by='volume_total_eur', ascending=False).head(10)

,agence,produit,annee,mois,nb_transactions,volume_total_eur,moyenne_transaction_eur
371,Bordeaux-Meriadeck,Credit Auto,2023,7,2,152605.80,76302.90
195,Nantes-Commerce,Assurance Vie,2024,10,1,150000.00,150000.00
242,Nantes-Commerce,Compte Epargne,2024,8,1,150000.00,150000.00
452,Paris-Centre,PEA,2024,2,2,146691.80,73345.90
468,Lyon-Part-Dieu,Compte Epargne,2022,9,3,144223.55,48074.52
199,Bordeaux-Meriadeck,Livret A,2023,6,2,101850.97,50925.49
289,Nantes-Commerce,Compte Epargne,2023,12,1,99999.99,99999.99
526,Lille-Grand-Place,Compte Courant,2023,12,2,99781.29,49890.65
42,Nantes-Commerce,Compte Courant,2023,11,2,94264.31,47132.16
225,Marseille-Vieux-Port,Credit Consommation,2022,5,2,91717.10,45858.55


Sous-requêtes : clients avec un solde inférieur à la moyenne nationale

In [71]:
# Query the view using the name defined in performance.sql
query_low_balance = "SELECT * FROM vw_clients_sous_moyenne;"
df_low_balance = pd.read_sql(query_low_balance, engine)
df_low_balance.head()

,client_id,score_credit_client,segment_client
0,CLI0060,645,Premium
1,CLI0034,457,Risque
2,CLI0085,390,Risque
3,CLI0009,645,Standard
4,CLI0031,559,Risque


CASE WHEN : calcul du taux de défaut par segment de risque

In [72]:
query_risk = "SELECT * FROM vw_taux_defaut_segment;"
df_risk = pd.read_sql(query_risk, engine)
df_risk

,segment_client,categorie_risque,total_transactions,nb_rejets,taux_defaut_pourcentage
0,Standard,Low,158,5,3.16
1,Premium,Medium,65,2,3.08
2,Risque,Medium,13,0,0.00
3,Standard,High,25,0,0.00
4,Risque,High,385,19,4.94
5,Premium,Low,286,22,7.69
6,Standard,Medium,1068,65,6.09


Jointures multi-tables

In [73]:
# Query the master view that joins: transactions, clients, agencies, products, categories, and dim_date
query_joins = "SELECT * FROM vw_transactions_full LIMIT 10;"
df_joins = pd.read_sql(query_joins, engine)

# Display the result to prove all tables are connected
df_joins

,transaction_id,client_id,segment_client,categorie_risque,agence,produit,categorie,date_transaction,annee,mois,montant,type_operation,devise,statut,is_anomaly
0,TXN000559,CLI0060,Premium,Medium,Marseille-Vieux-Port,Compte Epargne,Depot especes,2022-04-19 02:31:00,2022,4,2050.42,Credit,EUR,Complete,False
1,TXN001154,CLI0057,Risque,High,Inconnue,Credit Consommation,Retrait DAB,2024-06-20 20:51:00,2024,6,-123.66,Debit,GBP,Rejete,False
2,TXN000764,CLI0015,Standard,Medium,Lyon-Part-Dieu,PEA,Prelevement,2024-08-28 05:03:00,2024,8,-396.17,Debit,EUR,Complete,False
3,TXN001598,CLI0045,Standard,Low,Bordeaux-Meriadeck,Credit Consommation,Paiement CB,2024-01-07 08:16:00,2024,1,225.20,Credit,EUR,Complete,False
4,TXN001873,CLI0034,Risque,High,Bordeaux-Meriadeck,Credit Immobilier,Interets,2024-08-11 19:52:00,2024,8,935.32,Credit,EUR,Complete,False
5,TXN001864,CLI0061,Risque,High,Marseille-Vieux-Port,Compte Epargne,Remboursement credit,2024-04-28 23:08:00,2024,4,-256.73,Debit,CHF,Rejete,False
6,TXN000931,CLI0118,Standard,Medium,Lille-Grand-Place,Compte Epargne,Depot especes,2023-09-13 12:04:00,2023,9,2752.67,Credit,EUR,Rejete,False
7,TXN000620,CLI0005,Standard,Medium,Lille-Grand-Place,Livret A,Remboursement credit,2022-05-11 08:57:00,2022,5,1962.49,Credit,EUR,Complete,False
8,TXN000581,CLI0064,Premium,Low,Lille-Grand-Place,Compte Courant,Paiement CB,2022-10-24 11:30:00,2022,10,363.52,Credit,GBP,Complete,False
9,TXN000726,CLI0085,Risque,High,Toulouse-Capitole,Credit Consommation,Virement international,2022-04-21 21:15:00,2022,4,6451.32,Credit,EUR,Complete,True


Créer des vues analytiques pour les KPIs du dashboard (brief 3)

In [74]:
query_kpis = "SELECT * FROM vw_dashboard_kpis;"
df_kpis = pd.read_sql(query_kpis, engine)
df_kpis

,chiffre_affaires_total,panier_moyen,total_anomalies_detectees,nombre_clients_actifs
0,-236642.99,-137.025472,93,150
